# Lab 5, optional: train a GPT-2-sized model on one GPU

**For a demo. About 25 minutes on an A100, most of it one training run. Read the outputs; you do not need to follow the code.**

*Tested on October 7, 2026 on one A100 with PyTorch 2.11. The default run took 19 minutes of training, and 23 minutes for the whole notebook.*

The NanoGPT speedrun is a public contest: train a model about the size of GPT-2 small to a loss of 3.28 on held-back web text, using eight H100 GPUs, as fast as possible. The starting point in May 2024 took 45 minutes. The record on August 30, 2026 was under 40 seconds.

The current record cannot run here. It uses code written for the H100 and more memory than one Colab GPU has. This notebook runs an earlier record that is ordinary PyTorch:

| | |
|---|---|
| Record | number 11, November 10, 2024, by Brendan Hogan Rappazzo |
| Time on eight H100s | 7.2 minutes for 3,000 training steps |
| Model | 12 blocks, each token described by 768 numbers, about 162 million parameters |
| Text | FineWeb, a large collection of web pages, already cut into GPT-2 word pieces |

On one GPU the same recipe needs eight times as many GPU-minutes, and an A100 is slower than an H100. We measured 2.9 seconds per step on an A100, so the full 3,000 steps would take about two and a half hours. By default this notebook does a 400-step run, which takes about 19 minutes. Each step reads about half a million tokens.

*Drafted by Claude (Anthropic), based on the direction of Xiuye Chen. The training code is from [KellerJordan/modded-nanogpt](https://github.com/KellerJordan/modded-nanogpt).*

## 1. Settings

Choose a GPU first: *Runtime → Change runtime type → A100*. The cell picks how many sequences fit on the GPU at once. The recipe stays the same either way: the script adds up several passes before each update, so every step still covers 512 sequences of 1,024 tokens. With 32 sequences per pass, our test run used 18 GB of GPU memory.

In [ ]:
import os, torch
assert torch.cuda.is_available(), "No GPU. Choose one with Runtime > Change runtime type, then run this cell again."
gpu = torch.cuda.get_device_name(0)
memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

STEPS = 400        # 400 is the short demo, about 19 minutes on an A100. 3000 is the whole record run, about two and a half hours.
VAL_EVERY = 100    # measure the loss on held-back text every this many steps
DEVICE_BATCH = 64 if memory_gb > 70 else 32 if memory_gb > 30 else 16 if memory_gb > 20 else 8

os.environ.update(STEPS=str(STEPS), VAL_EVERY=str(VAL_EVERY), DEVICE_BATCH=str(DEVICE_BATCH))
os.makedirs("/content/speedrun", exist_ok=True)
%cd /content/speedrun
print(f"GPU: {gpu}, {memory_gb:.0f} GB")
print(f"{STEPS} steps, {DEVICE_BATCH} sequences per pass, {512 // DEVICE_BATCH} passes per step")
print(f"tokens read in training: {STEPS * 512 * 1024 / 1e6:.0f} million")
if "A100" not in gpu and "H100" not in gpu:
    print("Note: this was planned for an A100. On a smaller GPU it will be much slower.")

## 2. Get the record's code, and adapt it for one GPU

The speedrun stores the full source of every record inside that record's log file, so we take it from there. The cell prints each change it makes. None of them touches the model or the training recipe:

- the run length and batch size come from the settings above;
- two speed settings tuned for the contest's H100s are switched off;
- the model writes some text when training ends;
- no checkpoint file is saved.

In [ ]:
# Get the record's training script and adapt it for one GPU.
# The speedrun keeps every record's full source inside its log file, so we take it from there.
import os, urllib.request

COMMIT = "4ea6b937337a4889b8cfe3f38a93d120048d8f71"   # modded-nanogpt as of 2026-09-28, pinned so this notebook keeps working
LOG = ("https://raw.githubusercontent.com/KellerJordan/modded-nanogpt/" + COMMIT +
       "/records/track_1_short/2024-11-10_UNetDoubleLr/c87bb826-797b-4f37-98c7-d3a5dad2de74.txt")
text = urllib.request.urlopen(LOG).read().decode("utf-8")
bar = "=" * 100
src = text.split(bar)[1].lstrip("\n")          # the script sits between the first two rules of the log

def patch(old, new, what):
    global src
    assert src.count(old) == 1, "could not find the code to change: " + what
    src = src.replace(old, new)
    print("changed:", what)

# 1. Run length and batch size come from the settings cell above.
patch("    num_iterations : int = 3000 # number of iterations to run",
      "    num_iterations : int = int(os.environ.get('STEPS', 3000))", "number of training steps")
patch("    warmdown_iters : int = 900 # number of iterations of linear warmup/warmdown for triangular or trapezoidal schedule",
      "    warmdown_iters : int = int(0.3 * int(os.environ.get('STEPS', 3000)))  # the last 30% of the run, as in the record", "learning-rate wind-down")
patch("    val_loss_every : int = 125 # every how many steps to evaluate val loss? 0 for only at the end",
      "    val_loss_every : int = int(os.environ.get('VAL_EVERY', 125))", "how often to measure held-back loss")
patch("    device_batch_size : int = 64 # batch size, in sequences, per device",
      "    device_batch_size : int = int(os.environ.get('DEVICE_BATCH', 64))  # 64 needs an 80 GB GPU", "sequences per GPU pass")

# 2. Two speed settings tuned for the contest's H100s. One makes compilation take up to half an hour;
#    the other forces an attention kernel that may not be available here. Use PyTorch's defaults.
patch('if hasattr(config, "coordinate_descent_tuning"):\n    config.coordinate_descent_tuning = True # suggested by @Chillee\n',
      "", "slow compile-time tuning switched off")
patch("from torch.backends.cuda import enable_cudnn_sdp, enable_flash_sdp, enable_math_sdp, enable_mem_efficient_sdp\n"
      "enable_cudnn_sdp(True)\nenable_flash_sdp(False)\nenable_mem_efficient_sdp(False)\nenable_math_sdp(False)\n",
      "", "default attention kernel")

# 3. Let the model hand back its scores, so it can write at the end.
patch("    def forward(self, idx, target):\n\n        # forward the GPT model itself",
      "    def forward(self, idx, target=None):\n\n        # forward the GPT model itself", "model can be called without an answer key")
patch("        logits = logits.float()\n        loss = F.cross_entropy(",
      "        logits = logits.float()\n        if target is None:\n            return logits\n        loss = F.cross_entropy(", "model returns scores when writing")

# 4. Do not save a multi-gigabyte checkpoint; print a line every 10 steps instead of every step.
patch("S_{ii}' \\sim Uniform(0.5, 1.5)", "S_{ii}' ~ Uniform(0.5, 1.5)", "a note inside the code that made Python print a warning")
patch("        torch.save(log, 'logs/%s/state_step%06d.pt' % (run_id, step))", "        pass  # checkpoint not saved in this notebook", "no checkpoint file")
patch('        print(f"step:{step+1}/{args.num_iterations} train_loss:{train_loss.item():.4f} train_time:{approx_time:.0f}ms step_avg:{approx_time/timed_steps:.2f}ms")',
      '        if step < 10 or (step + 1) % 10 == 0:\n            print(f"step:{step+1}/{args.num_iterations} train_loss:{train_loss.item():.4f} train_time:{approx_time:.0f}ms step_avg:{approx_time/timed_steps:.2f}ms", flush=True)',
      "progress line every 10 steps")

# 5. When training ends, let the model write: the same loop as in the tour (scores, odds, draw one, append).
WRITE = '''
if master_process:
    import tiktoken
    enc = tiktoken.get_encoding("gpt2")
    net = getattr(raw_model, "_orig_mod", raw_model)   # the model itself, without the compiled wrapper
    net.eval()
    torch.manual_seed(1710)
    print("\\n" + "=" * 30 + " the model writes " + "=" * 30)
    for prompt in ["The capital of France is", "Once upon a time,", "Scientists announced on Tuesday that"]:
        idx = torch.tensor([enc.encode(prompt)], device=device)
        with torch.no_grad():
            for _ in range(60):
                scores = net(idx)[:, -1, :50257]                      # scores for the next token
                top_scores, top_tokens = (scores / 0.8).topk(50)       # keep the 50 most likely
                odds = torch.softmax(top_scores, dim=-1)
                nxt = top_tokens.gather(-1, torch.multinomial(odds, 1))  # draw one by those odds
                idx = torch.cat([idx, nxt], dim=1)                     # append, and go again
        print("---")
        print(enc.decode(idx[0].tolist()))
'''
patch("# -------------------------------------------------------------------------\n# clean up nice\ndist.destroy_process_group()",
      WRITE + "\n# -------------------------------------------------------------------------\n# clean up nice\ndist.destroy_process_group()", "writing at the end")

open("train_gpt2.py", "w").write(src)
compile(src, "train_gpt2.py", "exec")           # a syntax check; nothing runs yet
print("\nwrote train_gpt2.py:", src.count("\n"), "lines")

## 3. Get the text

FineWeb, already turned into GPT-2 word pieces. Each file holds 100 million tokens and is about 200 MB. The cell downloads only as many as this run will read. A warning about an HF token can be ignored.

In [ ]:
import math
from huggingface_hub import hf_hub_download

files_needed = math.ceil(STEPS * 512 * 1024 / 100_000_000) + 1
names = ["fineweb_val_000000.bin"] + [f"fineweb_train_{i:06d}.bin" for i in range(1, files_needed + 1)]
os.makedirs("data/fineweb10B", exist_ok=True)
for name in names:
    hf_hub_download(repo_id="kjj0/fineweb10B-gpt2", filename=name, repo_type="dataset", local_dir="data/fineweb10B")
print("downloaded:", len(names) - 1, "training files and 1 file of held-back text")

## 4. Train

This is the long cell. What to expect:

1. A pause of a minute or two while PyTorch compiles the model. Nothing prints during it.
2. `val_loss` at step 0, close to 10.8. That is the loss of a model that knows nothing: it is as unsure as someone choosing among all 50,304 tokens.
3. A `train_loss` line every 10 steps. `step_avg` is the time per step in milliseconds. Multiply it by the number of steps for the length of the run.
4. `val_loss` every 100 steps. This is the loss on held-back text, the number the contest is scored on.
5. At the end, the model continues three prompts.

In our test run on an A100 the held-back loss was 4.78 after 100 steps, 4.19 after 200, 3.97 after 300 and 3.77 after 400. Yours will differ a little, because the starting numbers are random.

In [ ]:
!pip -q install tiktoken
!torchrun --standalone --nproc_per_node=1 train_gpt2.py

## 5. The loss, as a picture

In [ ]:
import glob, re
import matplotlib.pyplot as plt

log = max(glob.glob("logs/*.txt"), key=os.path.getmtime)
lines = [l for l in open(log).read().split("=" * 100)[-1].splitlines() if l.startswith("step:")]
val = [(int(m.group(1)), float(m.group(2))) for l in lines if (m := re.match(r"step:(\d+)/\d+ val_loss:([\d.]+)", l))]
train = [(int(m.group(1)), float(m.group(2))) for l in lines if (m := re.match(r"step:(\d+)/\d+ train_loss:([\d.]+)", l))]
minutes = [float(m.group(1)) / 60000 for l in lines if (m := re.search(r"train_time:(\d+)ms", l))]

plt.figure(figsize=(8, 4.5))
plt.plot(*zip(*train), color="#9aa5b1", linewidth=1, label="training text, one batch")
plt.plot(*zip(*val), color="#d9480f", marker="o", linewidth=2, label="held-back text")
plt.axhline(3.28, color="#2f6f4f", linestyle="--", linewidth=1)
plt.text(0, 3.33, "the contest's target, 3.28", color="#2f6f4f")
plt.ylim(3, 11.2); plt.xlabel("training step (each reads about half a million tokens)"); plt.ylabel("loss (lower is better)")
plt.legend(frameon=False); plt.title("A GPT-2-sized model learning from web text"); plt.show()

print("held-back loss:", ", ".join(f"step {s}: {v:.2f}" for s, v in val))
print(f"time spent training: about {max(minutes):.0f} minutes")

## What to notice

- **It is the same recipe as the tiny model.** Guess the next token, measure the loss, nudge the numbers, repeat. The loss starts where blind guessing would put it and falls quickly at first, then slowly.
- **The text it writes** has real words and mostly real grammar, and little sense. The model has read about 200 million tokens. The models you call have read thousands of times more.
- **The loss is not at 3.28 yet.** That takes all 3,000 steps: set `STEPS = 3000` in the settings cell and run the notebook again, if you have about two and a half hours of GPU time.
- **Why the contest matters.** On the same eight GPUs, reaching 3.28 went from 45 minutes to under 40 seconds in a little over two years. The run you just watched is the version from November 2024, which took 7.2 minutes there.